#Project-3: Finsights Grey - RAG for Effective Information Retrieval



##Business Use Case





Problem Statement:


Finsights Grey Inc. is an innovative financial technology firm that specializes in providing advanced analytics and insights for investment management and financial planning. The company handles an extensive collection of 10-K reports from various industry players, which contain detailed information about financial performance, risk factors, market trends, and strategic initiatives. Despite the richness of these documents, Finsights Grey's financial analysts struggle with extracting actionable insights efficiently in a short span due to the manual and labor-intensive nature of the analysis. Going through the document to find the exact information needed at the moment takes too long. This bottleneck hampers the company's ability to deliver timely and accurate recommendations to its clients. To overcome these challenges, Finsights Grey Inc. aims to implement a Retrieval-Augmented Generation (RAG) model to automate the extraction, summarization, and analysis of information from the 10-K reports, thereby enhancing the accuracy and speed of their investment insights.


Objective:


As a Gen AI Data Scientist hired by Finsights Grey Inc., the objective is to develop an advanced RAG-based system to streamline the extraction and analysis of key information from 10-K reports. You are asked to deploy a Gradio app on HuggingFace spaces that can RAG 10-k reports and answer the questions of financial analysts swiftly.


The project will involve testing the RAG system on a current business problem. The Financial analysts are asked to research major cloud and AI platforms such as Amazon AWS, Google Cloud, Microsoft Azure, Meta AI, and IBM Watson to determine the most effective platform for this application. The primary goals include improving the efficiency of data extraction. Once the project is deployed, the system will be tested by a financial analyst with the following questions. Accurate text retrieval for these questions will imply the project's success.


Questions:




Has the company made any significant acquisitions in the AI space, and how are these acquisitions being integrated into the company's strategy?




How much capital has been allocated towards AI research and development?




What initiatives has the company implemented to address ethical concerns surrounding AI, such as fairness, accountability, and privacy?




How does the company plan to differentiate itself in the AI space relative to competitors?






Each Question must be asked for each of the five companies on the HuggingFace spaces.


By successfully developing this project, we aim to:


Improve the productivity of financial analysts by providing a competent tool.


Provide timely insights to improve client recommendations.


Strengthen FinTech Insights Inc.’s competitive edge by delivering more reliable and faster insights to clients.





Connect to a T4 GPU Instance to create the Vector Database.



##Setup



In [2]:
# Install the necessary libraries
# Same as Week 15 implementing_rag.
# tiktoken 0.6.0 throws an error for this Colab Python; 0.14.0 is the current wheel.
# !pip install -q openai==1.23.2 \
#                 tiktoken==0.14.0 \
#                 pypdf==4.0.1 \
#                 langchain==0.1.1 \
#                 langchain-community==0.0.13 \
#                 chromadb==0.4.22 \
#                 sentence-transformers==2.3.1

!pip install -q openai tiktoken==0.14.0 pypdf \
                "langchain>=0.2.0,<0.4.0" \
                "langchain-community>=0.2.0,<0.4.0" \
                chromadb sentence-transformers

#The error is colab versus colab noice. I have the libraries thatI require


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
langgraph-sdk 0.4.9 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.17 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.


In [3]:
#C2 Import the necessary Libraries

import json
import tiktoken
import pandas as pd
from openai import OpenAI

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

from google.colab import userdata, drive


##Impementing RAG

##Prepare Data

In [4]:
#C3 Upload Dataset-10k.zip and unzip it dataset folder using -d option
drive.mount('/content/drive')
!unzip -o "/content/drive/MyDrive/Colab Notebooks/Dataset-10k.zip" -d dataset
!ls -la dataset


Mounted at /content/drive
Archive:  /content/drive/MyDrive/Colab Notebooks/Dataset-10k.zip
  inflating: dataset/IBM-10-k-2023.pdf  
  inflating: dataset/Meta-10-k-2023.pdf  
  inflating: dataset/aws-10-k-2023.pdf  
  inflating: dataset/google-10-k-2023.pdf  
  inflating: dataset/msft-10-k-2023.pdf  
total 6980
drwxr-xr-x 2 root root    4096 Oct 28 18:56 .
drwxr-xr-x 1 root root    4096 Oct 28 18:56 ..
-rwxrwxr-- 1 root root  712683 Jun  7  2024 aws-10-k-2023.pdf
-rwxrwxr-- 1 root root  926655 Jun  7  2024 google-10-k-2023.pdf
-rwxrwxr-- 1 root root  574740 Jun  7  2024 IBM-10-k-2023.pdf
-rwxrwxr-- 1 root root 2481466 Jun  7  2024 Meta-10-k-2023.pdf
-rwxrwxr-- 1 root root 2434632 Jun  7  2024 msft-10-k-2023.pdf


##DB Creation

##Chunking



In [5]:
#C4 Provide pdf_folder_location
pdf_folder_location = "dataset"
print(pdf_folder_location)


dataset


In [6]:
#C5 Load the directory to pdf_loader
# Each PDF page becomes one LangChain Document; metadata includes source path and page
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)
reports = pdf_loader.load()
print(len(reports))
print(reports[0].metadata)
# 448 pages loaded


448
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-02-03T06:03:35-05:00', 'title': '0001018724-23-000004', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-02-03 for the period ending 2022-12-31', 'keywords': '0001018724-23-000004; ; 10-K', 'moddate': '2023-02-03T06:03:51-05:00', 'source': 'dataset/aws-10-k-2023.pdf', 'total_pages': 81, 'page': 0, 'page_label': '1'}


In [7]:
#C6 Create text_splitter using recursive splitter
# Token ruler = cl100k_base (same family as gpt-4). Cards of 512 tokens, 16-token overlap.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=16,
)
text_splitter


In [8]:
#C7 Create chunks
report_chunks = text_splitter.split_documents(reports)


In [9]:
#C8 Check the total number of chunks
print(len(report_chunks))
#902 cards from 448 pages


902


In [10]:
#C9 Check the first object in report_chunks and print it
print(report_chunks[0].page_content[:500])
print("---")
print(report_chunks[0].metadata)


Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 ____________________________________
FORM 10-K
____________________________________ 
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2022
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from            to             .
Commission File No. 000-22513
---
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-02-03T06:03:35-05:00', 'title': '0001018724-23-000004', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-02-03 for the period ending 2022-12-31', 'keywords': '0001018724-23-000004; ; 10-K', 'moddate': '2023-02-03T06:03:51-05:00', 'source': 'dataset/aws-10-k-2023.pdf', 'total_pages': 81, 'page': 0, 'page_label': '1'}


## Database Creation

In [11]:
#C10 Create a Colelction Name
collection_name = "finsights-10k-2023"
print(collection_name)


finsights-10k-2023


In [12]:
#C11 Initiate the embedding model 'thenlper/gte-large'
# Number-stamper. Runs on the T4. First run downloads the model (~0.7 GB).
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")
embedding_model


/tmp/ipykernel_533/3791093685.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
len(report_chunks)

902

In [14]:
#C12 Create the vector Database
persist_directory = "./reports_db"
vectorstore = Chroma.from_documents(
    documents=report_chunks,
    embedding=embedding_model,
    collection_name=collection_name,
    persist_directory=persist_directory,
)
vectorstore


In [15]:
#C13 Persist the DB
# Newer Chroma often already wrote the folder; persist() still matches the template.
vectorstore.persist()
!ls -la reports_db


total 20648
drwxr-xr-x 3 root root     4096 Oct 28 18:57 .
drwxr-xr-x 1 root root     4096 Oct 28 18:57 ..
drwxr-xr-x 2 root root     4096 Oct 28 18:57 96b5e4f3-ef3d-4fa0-91d7-0cc761dc021d
-rw-r--r-- 1 root root 21135360 Oct 28 18:57 chroma.sqlite3


/tmp/ipykernel_533/3250063988.py:3: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [16]:
#C14 Mount the Google Drive
#Drive mounted in Cell3 (C3)


In [17]:
#C15 Copy the persisted database to your drive

import shutil

drive_db = "/content/drive/MyDrive/Colab Notebooks/reports_db"
shutil.copytree("reports_db", drive_db, dirs_exist_ok=True)
!ls -la "/content/drive/MyDrive/Colab Notebooks/reports_db"
#21 MB and reports_db(cabinet) is on my G drive replicated to local drive.


total 20656
drwx------  4 root root     4096 Oct 28 18:57 .
drwx------ 17 root root     4096 Oct 30  2023 ..
drwx------  2 root root     4096 Sep 17 16:09 33212c06-83a5-40ca-a6f2-290aeb142d82
drwx------  2 root root     4096 Oct 28 18:57 96b5e4f3-ef3d-4fa0-91d7-0cc761dc021d
-rw-------  1 root root 21135360 Oct 28 18:57 chroma.sqlite3




Retrieve DB from GDrive
¶



## Set up CPU Instance

In [18]:
#C16 Install the required packages
#skipping...


In [19]:
#C17 Import the necessary Libraries
#skipping, 2026 logic makes this obsolete


## Set up Anyscale Credentials

In [20]:
#C18 get anyscale api key
# GL OpenAI proxy (Anyscale public API retired 1 Aug 2024)
openai_api_key = userdata.get("OPEN_API_KEY")


In [21]:
#C19 Initialise the client
client = OpenAI(
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    api_key=openai_api_key,
)


In [22]:
#C20 Provide the model name
model_name = "gpt-4"
print(model_name)


gpt-4


## Mount Google Drive

In [23]:
#C21 Mount the Google Drive
#Drive mounted in C3


## Load Vector DB from Google Drive

In [24]:
#C22 Initialise the embedding model
# Same  as C11; object still in this T4 session
embedding_model


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [25]:
#C23 Load the persisted DB
persisted_vectordb_location = "/content/drive/MyDrive/Colab Notebooks/reports_db"
print(persisted_vectordb_location)


/content/drive/MyDrive/Colab Notebooks/reports_db


In [26]:
#C24 Create a Colelction Name
collection_name = "finsights-10k-2023"
print(collection_name)


finsights-10k-2023


In [27]:
#C25 Load the persisted DB
reports_db = Chroma(
    persist_directory=persisted_vectordb_location,
    embedding_function=embedding_model,
    collection_name=collection_name,
)
print(reports_db._collection.count())
#902


902


/tmp/ipykernel_533/4292015335.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  reports_db = Chroma(


## Test your DB

In [28]:
#c26
user_question = (
    "How is the company integrating AI across their various business units, "
    "and what specific examples are provided in the reports of AI enhancing "
    "operational efficiencies or customer experiences?"
)
print(user_question)


How is the company integrating AI across their various business units, and what specific examples are provided in the reports of AI enhancing operational efficiencies or customer experiences?


In [29]:
# C27 Perform similarity search on the user_question
# You must add an extra parameter to the similarity search  function so that you can filter the response based on the 'source'  in the metadata of the doc
# The filter can be added as a parameter to the similarity search function
# This will allow you to retrieve chunks from a particular document
# Use the same format to filter your response based on the company.
# Note the format to add a filter. You must apply the same in your app.py file that you will upload on huggingface spaces

docs = reports_db.similarity_search(
    user_question,
    k=5,
    filter={"source": "dataset/google-10-k-2023.pdf"},
)
print(len(docs))
# 5 docs


5


In [30]:
#C28 Print the retrieved docs, their source and the page number
# (page number can be accessed using doc.metadata['page'] )

for i, doc in enumerate(docs, start=1):
    print(f"--- chunk {i} ---")
    print("source:", doc.metadata.get("source"))
    print("page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print()


--- chunk 1 ---
source: dataset/google-10-k-2023.pdf
page: 11
Our business environment is rapidly evolving and intensely competitive. Our businesses face changing 
technologies, shifting user needs, and frequent introductions of rival products and services. To compete successfully, 
we must accurately anticipate technology developments and deliver innovative, relevant and useful products, services, 
and technologies in a timely manner. As our businesses evol

--- chunk 2 ---
source: dataset/google-10-k-2023.pdf
page: 5
it can generalize and seamlessly understand, operate across, and combine different types of information, including 
text, code, audio, images, and video. Our teams across Alphabet will leverage Gemini, as well as other AI models we 
have previously developed and announced, across our business to deliver the best product and service experiences 
for our users, advertisers, partners, customers, and 

--- chunk 3 ---
source: dataset/google-10-k-2023.pdf
page: 8
• Collaborat

## RAG Q&A

## Prompt Design

In [31]:
# C29 Create a system message for the LLM
qna_system_message = """
You are a financial analyst assistant answering questions about one company's Form 10-K.
Use only the text inside ###Context.
If the context does not contain the answer, reply exactly: I don't know.
Do not use outside knowledge. Do not invent figures, acquisitions, or programs.
When you use a fact, mention the page number from the context if it is present.
"""
print(qna_system_message[:90])



You are a financial analyst assistant answering questions about one company's Form 10-K.



In [32]:
#C30 Create a message template
qna_user_message_template = """
###Context
{context}

###Question
{question}
"""
print(qna_user_message_template)



###Context
{context}

###Question
{question}



## Composing the response

In [33]:
#C31 Create a variable company to store the source of the context so that you can filter the similarity search
company = "dataset/aws-10-k-2023.pdf"


In [34]:
#C32 filter varible
relevant_document_chunks = reports_db.similarity_search(
    user_question,
    k=5,
    filter={"source": company},
)
context_for_query = "\n\n".join(
    f"[page {d.metadata.get('page')}]\n{d.page_content}"
    for d in relevant_document_chunks
)
print(context_for_query[:1500])
print("---")
print("n_chunks:", len(relevant_document_chunks))


[page 19]
Table of Contents
of suppliers, seasonality, and changes in payment and other terms over time, including the effect of balancing pricing and timing of payment terms with
suppliers.
We expect spending in technology and content will increase over time as we add computer scientists, designers, software and hardware engineers, and
merchandising employees. Our technology and content investment and capital spending projects often support a variety of product and service offerings due to
geographic expansion and the cross-functionality of our systems and operations. We seek to invest efficiently in several areas of technology and content,
including AWS, and expansion of new and existing product categories and service offerings, as well as in technology infrastructure to enhance the customer
experience and improve our process efficiencies. We believe that advances in technology, specifically the speed and reduced cost of processing power, data
storage and analytics, improved wireless

In [35]:
#C33 Fetch relevant documents and create context for query by joining page_content and page number of the retrieved docs

prompt = [
    {"role": "system", "content": qna_system_message},
    {
        "role": "user",
        "content": qna_user_message_template.format(
            context=context_for_query,
            question=user_question,
        ),
    },
]
print(prompt[0]["role"], len(prompt[1]["content"]))


print() # Print the whole context_for_query (after joining all the chunks. It should contain page number of every chunk)


system 12905



In [36]:
#C34 Craft the messages to pass to chat.completions.create
prompt = [
    {"role": "system", "content": qna_system_message},
    {
        "role": "user",
        "content": qna_user_message_template.format(
            context=context_for_query,
            question=user_question,
        ),
    },
]
print(prompt[0]["role"], len(prompt[1]["content"]))


system 12905


In [37]:
# #C35 Get a response from the LLM
# # Handle errors using try-except
# # print the content of the response

# try:
#     completion = client.chat.completions.create(
#         model=model_name,
#         messages=prompt,
#         temperature=0,
#         max_tokens=500,
#     )
#     answer = completion.choices[0].message.content
#     print(answer)
# except Exception as exc:
#     print(type(exc).__name__, ":", exc)


## C35 Host note — 17 Sep 2026

AuthenticationError : Error code: 401 - {'reason': 'Authentication failed: Batch inactive'}


C35 call to the Great Learning Olympus OpenAI proxy
(
https://aibe.mygreatlearning.com/openai/v1
, model gpt-4)
returned HTTP 401: Authentication failed: Batch inactive
after a fresh Settings → OpenAI Token.


Retrieval is unaffected.
Generation, LLM-as-judge, and the Gradio Space need another
OpenAI-compatible host.


Cell C35 commented out and proceeding on finding another  host.



## C35 Alternate host — paid OpenAI gpt-4o-mini

Since the planned host is not available I've elected to use the paid oen AI host 'gpt-4o-mini'

Same OpenAI Python client as the template (chat.completions.create).

Closest inexpensive stand-in for the school's gpt-4 slot.


In [38]:
# C36  point client at paid gpt-4o-mini
# Alternate host — paid OpenAI gpt-4o-mini (Olympus batch inactive)

HOST = "openai"

client = OpenAI(api_key=userdata.get("OPEN_API_KEY"))
model_name = "gpt-4o-mini"
rater_model = "gpt-4o-mini"

print("HOST:", HOST, "model:", model_name)


HOST: openai model: gpt-4o-mini


In [39]:
#C37 Connect to alternate model

try:
    completion = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0,
        max_tokens=500,
    )
    answer = completion.choices[0].message.content
    print(answer)
except Exception as exc:
    print(type(exc).__name__, ":", exc)

#It used the cards and refused to invent examples. Shop works.


The company is integrating AI by investing in technology infrastructure to enhance the customer experience and improve process efficiencies. Specific examples include the use of artificial intelligence and machine learning to improve users’ experience on the Internet and increase its ubiquity in people’s lives. Additionally, the company is investing in AWS, which offers a broad set of on-demand technology services, including machine learning, to developers and enterprises of all sizes (page 19).


## Evaluation


## Craft prompts for evaluation

In [40]:
#C38 Confirm the rater model from C36
# Pick a model that's offers more performace as a rater_model. Most of the time a model with more parameters is more performant.
# Commercial model paid for with pernal $

rater_model = "gpt-4o-mini"
print(rater_model)


gpt-4o-mini


In [41]:
#C39 Create a prompt for the rater LLM to check the groundedness of the response

groundedness_rater_system_message = """
You are an evaluator. Score how well the ANSWER is grounded in the CONTEXT.
Grounded means every factual claim in the answer appears in the context.
Ignore whether the answer is complete. Only judge support from context.

Return exactly two lines:
Score: <integer 1 to 5>
Reason: <one short sentence>
"""
print(groundedness_rater_system_message[:60])



You are an evaluator. Score how well the ANSWER is grounded


In [42]:
#C40 Create a prompt for the rater LLM to check the relevance of the response
relevance_rater_system_message = """
You are an evaluator. Score how well the ANSWER addresses the QUESTION.
Relevant means it answers what was asked. Ignore extra style.
If the answer correctly says it does not know because the context lacks
the fact, that can still be relevant.

Return exactly two lines:
Score: <integer 1 to 5>
Reason: <one short sentence>
"""
print(relevance_rater_system_message[:60])



You are an evaluator. Score how well the ANSWER addresses t


In [43]:
#C41 Create user message template such that question, answer and context can be provided through it.
user_message_template = """
###Question
{question}

###Answer
{answer}

###Context
{context}
"""
print(user_message_template)



###Question
{question}

###Answer
{answer}

###Context
{context}



## Test the Evaluation on One Sample

In [44]:
#C42
user_input = "How much is the company investing in research and development, and what are the key areas of focus for innovation?"


In [45]:
#C43 Fetch relevant documents and create context for query by joining page_content and page number of the retrieved docs

relevant_document_chunks = reports_db.similarity_search(
    user_input,
    k=5,
    filter={"source": company},
)
context_for_query = "\n\n".join(
    f"[page {d.metadata.get('page')}]\n{d.page_content}"
    for d in relevant_document_chunks
)
print("n_chunks:", len(relevant_document_chunks))
print(context_for_query[:400])


n_chunks: 5
[page 19]
Table of Contents
of suppliers, seasonality, and changes in payment and other terms over time, including the effect of balancing pricing and timing of payment terms with
suppliers.
We expect spending in technology and content will increase over time as we add computer scientists, designers, software and hardware engineers, and
merchandising employees. Our technology and content investmen


In [46]:
#C44 Create the messages for chat.completion.create()


prompt = [
    {"role": "system", "content": qna_system_message},
    {
        "role": "user",
        "content": qna_user_message_template.format(
            context=context_for_query,
            question=user_input,
        ),
    },
]
print(prompt[0]["role"], len(prompt[1]["content"]))


system 10780


In [47]:
#C45 Get a response from the LLM
# Handle errors using try-except

try:
    completion = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0,
        max_tokens=500,
    )
    answer = completion.choices[0].message.content
    print(answer)
except Exception as exc:
    print(type(exc).__name__, ":", exc)


I don't know.


In [48]:
#C46 Create messages for groundness LLM

groundedness_messages = [
    {"role": "system", "content": groundedness_rater_system_message},
    {
        "role": "user",
        "content": user_message_template.format(
            question=user_input,
            answer=answer,
            context=context_for_query,
        ),
    },
]
print(len(groundedness_messages[1]["content"]))


10805


In [49]:
#C47 Print the response of the rater LLM on groundedness

try:
    groundedness_completion = client.chat.completions.create(
        model=rater_model,
        messages=groundedness_messages,
        temperature=0,
        max_tokens=150,
    )
    groundedness_evaluation = groundedness_completion.choices[0].message.content
    print(groundedness_evaluation)
except Exception as exc:
    print(type(exc).__name__, ":", exc)


Score: 1  
Reason: The answer does not provide any information from the context regarding investment in research and development or areas of focus for innovation.


In [50]:
#C48 Print the response of the rater LLM on relevance

relevance_messages = [
    {"role": "system", "content": relevance_rater_system_message},
    {
        "role": "user",
        "content": user_message_template.format(
            question=user_input,
            answer=answer,
            context=context_for_query,
        ),
    },
]
try:
    relevance_completion = client.chat.completions.create(
        model=rater_model,
        messages=relevance_messages,
        temperature=0,
        max_tokens=150,
    )
    relevance_evaluation = relevance_completion.choices[0].message.content
    print(relevance_evaluation)
except Exception as exc:
    print(type(exc).__name__, ":", exc)


Score: 1  
Reason: The answer does not provide any relevant information regarding the company's investment in research and development or its key areas of focus.


In [51]:
#C49 Print the response of the rater LLM on relevance
print(relevance_evaluation)


Score: 1  
Reason: The answer does not provide any relevant information regarding the company's investment in research and development or its key areas of focus.


## Evaluation on multiple-queries

In [52]:
#C50 List of queries
queries = [ "What are the company’s policies and frameworks regarding AI ethics, governance, and responsible AI use as detailed in their 10-K reports?",
           "What are the primary business segments of the company, and how does each segment contribute to the overall revenue and profitability?",
            "What are the key risk factors identified in the 10-K report that could potentially impact the company’s business operations and financial performance?"]

#C50.1 Create a DataFrame to store the results
df = pd.DataFrame(
    columns=[
        "query",
        "response",
        "context",
        "groundedness_evaluation",
        "relevance_evaluation",
    ]
)
print(df.shape)

#C50.2 run a loop to get answer for every query and every company and then rate them on groundedness and relevance
#Generate 15 queries( 5 companies x 3 queries) and store
companies = ["google", "aws", "msft", "IBM", "Meta"]
rows = []

# two loops 15 entries
for company_key in companies:
    source = f"dataset/{company_key}-10-k-2023.pdf"
    for query in queries:
        docs = reports_db.similarity_search(query, k=5, filter={"source": source})
        context = "\n\n".join(
            f"[page {d.metadata.get('page')}]\n{d.page_content}" for d in docs
        )
        rows.append(
            {
                "query": query,
                "response": "",
                "context": context,
                "groundedness_evaluation": "",
                "relevance_evaluation": "",
            }
        )

df = pd.DataFrame(rows)
print(df.shape)


#C50.3 store the query, response, context,groundedness_evaluation, relevance_evaluation in a dataframe
#Capture responses
responses = []
for i, row in df.iterrows():
    gen_messages = [
        {"role": "system", "content": qna_system_message},
        {
            "role": "user",
            "content": qna_user_message_template.format(
                context=row["context"],
                question=row["query"],
            ),
        },
    ]
    try:
        gen = client.chat.completions.create(
            model=model_name,
            messages=gen_messages,
            temperature=0,
            max_tokens=500,
        )
        responses.append(gen.choices[0].message.content)
    except Exception as exc:
        responses.append(f"ERROR: {type(exc).__name__}")
    print("generated", i + 1, "of", len(df))

df["response"] = responses
print(df["response"].str[:80].to_string())

#50.4 Capture Groundedness
ground_scores = []
for i, row in df.iterrows():
    rate_user = user_message_template.format(
        question=row["query"],
        answer=row["response"],
        context=row["context"],
    )
    try:
        g = client.chat.completions.create(
            model=rater_model,
            messages=[
                {"role": "system", "content": groundedness_rater_system_message},
                {"role": "user", "content": rate_user},
            ],
            temperature=0,
            max_tokens=150,
        )
        ground_scores.append(g.choices[0].message.content)
    except Exception as exc:
        ground_scores.append(f"ERROR: {type(exc).__name__}")
    print("grounded", i + 1, "of", len(df))

df["groundedness_evaluation"] = ground_scores
print(df["groundedness_evaluation"].str[:80].to_string())

#C50.5 Capture relevance (last column)
rel_scores = []
for i, row in df.iterrows():
    rate_user = user_message_template.format(
        question=row["query"],
        answer=row["response"],
        context=row["context"],
    )
    try:
        r = client.chat.completions.create(
            model=rater_model,
            messages=[
                {"role": "system", "content": relevance_rater_system_message},
                {"role": "user", "content": rate_user},
            ],
            temperature=0,
            max_tokens=150,
        )
        rel_scores.append(r.choices[0].message.content)
    except Exception as exc:
        rel_scores.append(f"ERROR: {type(exc).__name__}")
    print("relevant", i + 1, "of", len(df))

df["relevance_evaluation"] = rel_scores
print(df["relevance_evaluation"].str[:80].to_string())


(0, 5)
(15, 5)
generated 1 of 15
generated 2 of 15
generated 3 of 15
generated 4 of 15
generated 5 of 15
generated 6 of 15
generated 7 of 15
generated 8 of 15
generated 9 of 15
generated 10 of 15
generated 11 of 15
generated 12 of 15
generated 13 of 15
generated 14 of 15
generated 15 of 15
0     The company has devoted significant resources ...
1     The primary business segments of the company a...
2     The key risk factors identified in the 10-K re...
3                                         I don't know.
4     The company has organized its operations into ...
5     The key risk factors identified in the 10-K re...
6     The company has developed a responsible AI pra...
7     The primary business segments of the company a...
8     The key risk factors identified in the 10-K re...
9                                         I don't know.
10    The primary business segments of the company a...
11    The key risk factors identified in the 10-K re...
12                                   

Twelve of fifteen answers scored 5 on groundedness.

Three ethics questions returned I don't know and scored 1 on groundedness because the retrieved cards did not mention AI ethics policies.

Relevance stayed high (4–5).

Host is personal gpt-4o-mini

In [53]:
#C51 Your Dataframe should have 15 rows - 3 queries for each of 5 companies - 3*5 = 15
# Show the top 10 rows of the dataframe
df.head(10)


,query,response,context,groundedness_evaluation,relevance_evaluation
0,What are the company’s policies and frameworks...,The company has devoted significant resources ...,"[page 16]\nsocial media or other platforms, wh...",Score: 5 \nReason: The answer accurately refl...,Score: 5 \nReason: The answer provides a comp...
1,What are the primary business segments of the ...,The primary business segments of the company a...,[page 33]\nTAC as a percentage of revenues gen...,Score: 5 \nReason: The answer accurately refl...,Score: 5 \nReason: The answer clearly identif...
2,What are the key risk factors identified in th...,The key risk factors identified in the 10-K re...,"[page 30]\ninvestigations, and other forms of ...",Score: 5 \nReason: All key risk factors menti...,Score: 5 \nReason: The answer comprehensively...
3,What are the company’s policies and frameworks...,I don't know.,[page 13]\nservices or access to our facilitie...,Score: 1 \nReason: The answer does not refere...,Score: 5 \nReason: The answer directly addres...
4,What are the primary business segments of the ...,The company has organized its operations into ...,[page 65]\nTable of Contents\nNote 10 — SEGMEN...,Score: 5 \nReason: Every factual claim in the...,Score: 5 \nReason: The answer thoroughly deta...
5,What are the key risk factors identified in th...,The key risk factors identified in the 10-K re...,[page 18]\nTable of Contents\nItem 7. Manageme...,Score: 5 \nReason: The answer accurately refl...,Score: 5 \nReason: The answer directly lists ...
6,What are the company’s policies and frameworks...,The company has developed a responsible AI pra...,[page 5]\n5 \nto launch a Generative AI Skills...,Score: 5 \nReason: The answer is fully suppor...,Score: 5 \nReason: The answer thoroughly deta...
7,What are the primary business segments of the ...,The primary business segments of the company a...,[page 16]\nof our key businesses. The segments...,Score: 5 \nReason: The answer accurately refl...,Score: 5 \nReason: The answer thoroughly deta...
8,What are the key risk factors identified in th...,The key risk factors identified in the 10-K re...,[page 10]\n10 \nNote About Forward-Looking Sta...,Score: 5 \nReason: The answer accurately refl...,Score: 5 \nReason: The answer directly identi...
9,What are the company’s policies and frameworks...,I don't know.,[page 13]\nreference.\nItem 7A. Quantitative a...,Score: 1 \nReason: The answer does not refere...,Score: 5 \nReason: The answer directly acknow...


You might experience some hallucination in LLM's response. Try to change your prompt to mitigate this. Selecting a good model will also help mitigating hallucination, increase groundedness and relevance.

No extra prompt pass. gpt-4o-mini + "I don't know" already kept
unsupported claims out of the 15-row sheet.


The three groundedness-1 rows are empty ethics answers


no invented facts.

## Gradio Interface


In [54]:
#C52
%%writefile app.py


# C52.1 Setup Import the necessary Libraries
import os
import json
import uuid
from pathlib import Path

from openai import OpenAI
import gradio as gr

from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain_community.vectorstores import Chroma
from huggingface_hub import CommitScheduler

print("imports ok")

#C52.2 — Create Client (Colab test)
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
model_name = "gpt-4o-mini"
print(model_name)

#C52.3 — Define the embedding model and the vectorstore
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")
persisted_vectordb_location = "/content/drive/MyDrive/Colab Notebooks/reports_db"
collection_name = "finsights-10k-2023"

vectorstore_persisted = Chroma(
    persist_directory=persisted_vectordb_location,
    embedding_function=embedding_model,
    collection_name=collection_name,
)
print(vectorstore_persisted._collection.count())



#C52.4 Load the persisted vectorDB
#Uses embedding_model from C11 or C52.3
persisted_vectordb_location = "/content/drive/MyDrive/Colab Notebooks/reports_db"
collection_name = "finsights-10k-2023"

vectorstore_persisted = Chroma(
    persist_directory=persisted_vectordb_location,
    embedding_function=embedding_model,
    collection_name=collection_name,
)
print(vectorstore_persisted._collection.count())

#C52.5 Prepare the logging functionality
#log into huggingfac, create a datset, scott-j-horning/finsights-10k-logs

log_file = Path("logs/") / f"data_{uuid.uuid4()}.json"
log_folder = log_file.parent

scheduler = CommitScheduler(
    repo_id="scott-j-horning/finsights-10k-logs",
    repo_type="dataset",
    folder_path=log_folder,
    path_in_repo="data",
    every=2
)

print(log_file)
print(log_folder.exists())
print(type(scheduler))

#C52.6 Define the Q&A system message
qna_system_message = """
You are a financial analyst assistant answering questions about one company's Form 10-K.
Use only the text inside ###Context.
If the context does not contain the answer, reply exactly: I don't know.
Do not use outside knowledge. Do not invent figures, acquisitions, or programs.
When you use a fact, mention the page number from the context if it is present.
"""
print(qna_system_message[:80])

#C52.7 Define the user message template
qna_user_message_template = """
###Context
{context}

###Question
{question}
"""
print(qna_user_message_template)



#C52.8 Define the predict function that runs when 'Submit' is clicked or when a API request is made
def predict(user_input,company):

    filter = "dataset/"+company+"-10-k-2023.pdf"
    relevant_document_chunks = vectorstore_persisted.similarity_search(user_input, k=5, filter={"source":filter})

    #C52.8.1 Create context_for_query
    context_for_query = "\n\n".join(
        f"[page {d.metadata.get('page')}]\n{d.page_content}"
        for d in relevant_document_chunks
    )

    #C52.8.2 Create messages
    messages = [
        {"role": "system", "content": qna_system_message},
        {
            "role": "user",
            "content": qna_user_message_template.format(
                context=context_for_query,
                question=user_input,
            ),
        },
    ]

    #C52.8.3 Get response from the LLM
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=0,
            max_tokens=500,
        )
        prediction = completion.choices[0].message.content
    except Exception as exc:
        prediction = f"ERROR: {type(exc).__name__}"

    # While the prediction is made, log both the inputs and outputs to a local log file
    # While writing to the log file, ensure that the commit scheduler is locked to avoid parallel
    # access

    with scheduler.lock:
        with log_file.open("a") as f:
            f.write(json.dumps(
                {
                    'user_input': user_input,
                    'retrieved_context': context_for_query,
                    'model_response': prediction
                }
            ))
            f.write("\n")

    return prediction

#C52.9 Set-up the Gradio UI
# Add text box and radio button to the interface
# The radio button is used to select the company 10k report in which the context needs to be retrieved.

textbox = gr.Textbox(
    label="Question",
    placeholder="Ask about the selected company's 10-K",
)
company = gr.Radio(
    choices=["google", "aws", "msft", "IBM", "Meta"],
    label="Company 10-K",
    value="google",
)

#C52.10 Create the interface
# For the inputs parameter of Interface provide [textbox,company]
demo = gr.Interface(
    fn=predict,
    inputs=[textbox, company],
    outputs=gr.Textbox(label="Answer"),
    title="Finsights Grey — 10-K RAG",
)
demo.queue()
demo.launch(share=True)


Writing app.py


In [55]:
#C53 Writing requirements.txt.

%%writefile requirements.txt
openai
gradio
langchain-community
chromadb
sentence-transformers
huggingface_hub


Writing requirements.txt


Paste your gradio app link and logs link

Deployment links

App (public Space):

https://huggingface.co/spaces/scott-j-horning/finsights-10k-rag

Logs (public Dataset):

https://huggingface.co/datasets/scott-j-horning/finsights-10k-logs


Host: personal OpenAI gpt-4o-mini